<a href="https://colab.research.google.com/github/9lana/WAY-AI/blob/main/neural_net.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
import torch
import torch.nn as nn
import json

In [17]:
training_data = [
      ("this is amazing", 1),
          ("i love this product", 1),
              ("absolutely wonderful experience", 1),
                  ("great quality and fast delivery", 1),
                      ("i am very happy with this", 1),
                          ("excellent work keep it up", 1),
                              ("best purchase i ever made", 1),
                                  ("fantastic service highly recommend", 1),
                                      ("really good and very useful", 1),
                                          ("superb quality loved it", 1),
                                              ("this is terrible", 0),
                                                  ("i hate this so much", 0),
                                                      ("worst experience ever", 0),
                                                          ("absolutely awful and disgusting", 0),
                                                              ("very bad quality do not buy", 0),
                                                                  ("horrible service never again", 0),
                                                                      ("complete waste of money", 0),
                                                                          ("disappointing and broken on arrival", 0),
                                                                              ("really poor and useless product", 0),
                                                                                  ("garbage total garbage", 0)
                                                       ]


In [18]:
def tokenize(sentence):
  return sentence.lower().split()

In [19]:
def build_vocab(data):
  vocab = {}
  idx = 0
  for sentence, _ in data:
    for word in tokenize(sentence):
      if word not in vocab:
        vocab[word] = idx
        idx += 1
  return vocab

In [20]:
def bag_of_word(sentence, vocab):
  vector = [0.0] * len(vocab)
  for word in tokenize(sentence):
    if word in vocab:
      vector[vocab[word]] += 1.0
  return vector


In [21]:
vocab = build_vocab(training_data)
vocab_size = len(vocab)
print(f"{vocab}")

{'this': 0, 'is': 1, 'amazing': 2, 'i': 3, 'love': 4, 'product': 5, 'absolutely': 6, 'wonderful': 7, 'experience': 8, 'great': 9, 'quality': 10, 'and': 11, 'fast': 12, 'delivery': 13, 'am': 14, 'very': 15, 'happy': 16, 'with': 17, 'excellent': 18, 'work': 19, 'keep': 20, 'it': 21, 'up': 22, 'best': 23, 'purchase': 24, 'ever': 25, 'made': 26, 'fantastic': 27, 'service': 28, 'highly': 29, 'recommend': 30, 'really': 31, 'good': 32, 'useful': 33, 'superb': 34, 'loved': 35, 'terrible': 36, 'hate': 37, 'so': 38, 'much': 39, 'worst': 40, 'awful': 41, 'disgusting': 42, 'bad': 43, 'do': 44, 'not': 45, 'buy': 46, 'horrible': 47, 'never': 48, 'again': 49, 'complete': 50, 'waste': 51, 'of': 52, 'money': 53, 'disappointing': 54, 'broken': 55, 'on': 56, 'arrival': 57, 'poor': 58, 'useless': 59, 'garbage': 60, 'total': 61}


In [22]:
idk = []
for s,_ in training_data:
  huh = bag_of_word(s, vocab)
  idk.append(huh)

X = torch.tensor(idk, dtype=torch.float32)
print(f"{X}")

why = []
for _, label in training_data:
  huh = float(label)
  why.append(huh)
Y1 = torch.tensor(why, dtype=torch.float32)
Y = Y1.unsqueeze(1)
print(f"{Y}")


tensor([[1., 1., 1.,  ..., 0., 0., 0.],
        [1., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 1., 0., 0.],
        [0., 0., 0.,  ..., 0., 2., 1.]])
tensor([[1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.],
        [0.]])


In [23]:
HIDDEN_SIZE = 8
class SentimentNet(nn.Module):
  def __init__(self, vocab_size, hidden_size):
    super().__init__()

    self.layer1 = nn.Linear(vocab_size, hidden_size)
    self.relu = nn.ReLU()
    self.layer2 = nn.Linear(hidden_size, 1)
    self.sigmoid = nn.Sigmoid()

  def forward(self, x):
    out = self.layer1(x)
    out = self.relu(out)
    out = self.layer2(out)
    out = self.sigmoid(out)

    return out



In [24]:
model = SentimentNet(vocab_size, HIDDEN_SIZE)
criterion = nn.BCELoss()
optimiser = torch.optim.SGD(model.parameters(), lr=0.5)

In [25]:
epochs = 300
for epoch in range(epochs):
  predicted = model(X)
  l = criterion(predicted, Y)
  optimiser.zero_grad()
  l.backward()
  optimiser.step()



In [26]:
torch.save(model.state_dict(), "model_nn.pth")
with open("vocab.json", "w") as f:
  json.dump(vocab, f)



In [27]:
loaded_module = SentimentNet(vocab_size, HIDDEN_SIZE)
weight = torch.load("model_nn.pth", weights_only=True)
loaded_module.load_state_dict(weight)
loaded_module.eval()


SentimentNet(
  (layer1): Linear(in_features=62, out_features=8, bias=True)
  (relu): ReLU()
  (layer2): Linear(in_features=8, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)

In [28]:
def predict(sentence):
  x = bag_of_word(sentence, vocab)
  x_tensor = torch.tensor(x, dtype=torch.float32)
  x_tensor.unsqueeze(0)
  with torch.no_grad():
    prob = loaded_module(x_tensor).item()
  if prob >= 0.5:
    label = "GOOD"
    return label, prob
  else:
    label = "BAD"
    return label, prob



Test

In [30]:
while True:
  try:
    user_input = input("Enter something")
    if not user_input:
      continue
    label, prob = predict(user_input)
    print(f"{label} {prob}\n")
  except KeyboardInterrupt:
    print("aight (>_<)")
    break

Enter somethingi love you
GOOD 0.9880808591842651

Enter somethingi hate you
BAD 0.3131861090660095

Enter somethingi fucking love you
GOOD 0.9880808591842651

aight (>_<)
